# 06e · 브라운 다리, 드리프트, 기하 브라운 운동 (Brownian Bridge, Drift and Geometric Brownian Motion)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Lawler 2e §8.7 (스케일링과 브라운 다리), §8.8 (드리프트 BM) · Ross 12e §10.3 (드리프트 BM, 기하 BM), §10.5 (드리프트 BM 의 최댓값) · Shreve II §4.7 (브라운 다리: 가우스 과정·조건화), §3.2.7 (이항 → 로그정규 극한), §5.2 (Girsanov — 인용만) · G&S 4e §13.6 (excursions and the Brownian bridge) · Karlin & Taylor 2e §7.4 (드리프트, 기하 BM 등 변형) · Durrett 3e §6 (이항 모형 → Black–Scholes; GBM 은 여기서만) |
| 선수 노트북 | 06d (지수 마팅게일, 드리프트 BM 의 한쪽 도달 확률, 다리 교차 보정), 06c (반사 원리, $(M_t,B_t)$ 결합분포, 격자 사이의 다리 보정), 06b (가우스 과정, $\mathrm{Cov}(B_s,B_t)=\min(s,t)$, $dB^2=dt$) |
| 예상 소요 | 90분 |
| 상태 | draft |

06c–06d 에서 세 대상이 이름 없이 등장했다: 격자 사이를 채운 **브라운 다리**, 지수 마팅게일이 다룬 **드리프트 BM**, 그리고 $e^{\theta B_t-\theta^2t/2}$ 자체 — 정규화하면 **기하 브라운 운동**(GBM) 이다. 이 노트북은 셋을 각각 하나의 과정으로 정식화한다. 다리는 "도착지를 아는 BM" 이라 공분산이 $\min(s,t)-st/T$ 로 줄고 $B_T$ 와 독립이다 (그래서 06c 의 격자 보정이 정확했다). 드리프트 BM 은 대칭이 깨져 반사 원리를 직접 못 쓰지만, 06d 의 지수 마팅게일을 '환율' 로 쓰면 최댓값·도달시간 분포(역가우스) 가 나온다. GBM 은 로그수익률이 BM 인 모형인데, 평균 $S_0e^{\mu t}$ 와 전형적 경로 $S_0e^{(\mu-\sigma^2/2)t}$ 가 갈라지는 **변동성 끌림** 이 오늘의 마지막 주제다 — 그리고 그 로그정규 꼬리가 표본평균 MC 를 무력하게 만드는 것을 직접 본다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm, invgauss, lognorm
from scipy.integrate import quad
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths, brownian_bridge, gbm_paths, running_max
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("06e")
setup_plots()
N_PATHS = scale(20_000)              # 대부분의 실험; FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_BIG = scale(100_000)               # 조건화(채택-기각) 실험: 채택률 8% 라 10 배 크게
N_GBM = scale(20_000, divisor=5)     # GBM 주변분포 (싸므로 FAST 에서도 4000)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_BIG={N_BIG}, N_GBM={N_GBM}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (06d) $B_0=0$ 에서 $(-a,b)$ 탈출 확률과 평균 탈출시간은?

<details><summary>정답</summary>

$P(B_\tau=b)=a/(a+b)$, $E\tau=ab$ ($\sigma=1$; 일반 $\sigma$ 에서는 $ab/\sigma^2$). 두 마팅게일 $B_t$, $B_t^2-t$ 에 선택적 정지 정리를 건 결과다.

</details>

**Q2.** (06d) 드리프트 $\mu<0$ 인 BM $X_t=\mu t+\sigma B_t$ 가 $b>0$ 에 도달할 확률과 $\sup_tX_t$ 의 분포는?

<details><summary>정답</summary>

$P(T_b<\infty)=e^{-2|\mu|b/\sigma^2}$; 따라서 $\sup_tX_t\sim\mathrm{Exp}(2|\mu|/\sigma^2)$, 평균 $\sigma^2/(2|\mu|)$. 오늘 정리 2 의 유한 지평 공식에서 $t\to\infty$ 극한으로 다시 나온다.

</details>

**Q3.** (06d) 지수 마팅게일 $e^{\theta B_t-\theta^2t/2}$ 에서 시간항이 상쇄되는 이유는?

<details><summary>정답</summary>

정규 mgf $E[e^{\theta D}]=e^{\theta^2(t-s)/2}$ ($D=B_t-B_s$) 가 정확히 지수의 시간항 $e^{-\theta^2(t-s)/2}$ 를 지운다. 오늘 GBM 의 $-\sigma^2t/2$ 보정항이 같은 mgf 에서 나온다.

</details>

**Q4.** (06c) 도달시간 $T_a$ 는 어떤 단순 확률변수와 같은 분포이며 평균·중앙값은?

<details><summary>정답</summary>

$T_a\overset d=a^2/Z^2$ ($Z\sim N(0,1)$); 평균 $\infty$, 중앙값 $a^2/z_{0.75}^2\approx2.198a^2$. 오늘 드리프트 $\mu>0$ 을 넣으면 $T_a$ 가 평균 $a/\mu$ 의 역가우스 분포가 되어 평균이 유한해진다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **표준 브라운 다리**(Brownian bridge; $[0,T]$, $0\to0$): $X_t=B_t-\frac tTB_T$. 일반 끝점 $a\to b$: $X_t+a+(b-a)\frac tT$ (`brownian_bridge(..., a, b)`). $X_0=X_T=0$, 가우스 과정, 평균 0, $\mathrm{Cov}(X_s,X_t)=\min(s,t)-\frac{st}T$, 특히 $\mathrm{Var}X_t=\frac{t(T-t)}T$ (최대 $T/4$, $t=T/2$ 에서).
- **조건화 표현**: $(B_t)_{t\le T}\mid B_T=y\ \overset d=\ X_t+\frac tTy$ — "$B_T$ 를 알면 남는 불확실성이 다리" 다. 06c 의 격자 보정은 이 사실을 각 격자 구간에 적용한 것이다: 격자값 $(x,y)$ 가 주어지면 구간 안은 (같은 $\Delta t$ 의) 브라운 다리이고 그 최댓값 CDF 가 $e^{-2(z-x)(z-y)/\Delta t}$ 다 (정리 1 따름정리).
- **드리프트 BM**(Brownian motion with drift): $X_t=x_0+\mu t+\sigma B_t$; 러닝 최댓값 $M_t=\max_{s\le t}X_s$, 도달시간 $T_a=\inf\{t:X_t=a\}$. 사건 $\{M_t\ge a\}=\{T_a\le t\}$ (경로 연속성).
- **역가우스 분포**(inverse Gaussian) $\mathrm{IG}(m,\lambda)$: 밀도 $\sqrt{\dfrac{\lambda}{2\pi t^3}}\exp\Big(-\dfrac{\lambda(t-m)^2}{2m^2t}\Big)$ ($t>0$), 평균 $m$, 분산 $m^3/\lambda$. $\mu>0$ 이면 $T_a\sim\mathrm{IG}(a/\mu,\ a^2/\sigma^2)$ (scipy: `invgauss(mu=m/lam, scale=lam)`). $\mu\to0$ 이면 $m\to\infty$ 이고 밀도는 $\frac{a}{\sigma\sqrt{2\pi t^3}}e^{-a^2/(2\sigma^2t)}$ 로 돌아간다 ($\sigma=1$ 이면 06c 의 $\frac{a}{\sqrt{2\pi t^3}}e^{-a^2/2t}$).
- **기하 브라운 운동**(geometric Brownian motion, GBM): $S_t=S_0\exp\big((\mu-\sigma^2/2)t+\sigma B_t\big)$; $\log S_t\sim N\big(\log S_0+(\mu-\sigma^2/2)t,\ \sigma^2t\big)$ (로그정규). 07b 에서 $dS=\mu S\,dt+\sigma S\,dB$ 의 해임을 이토 공식으로 보인다. `gbm_paths` 는 정확한 로그정규 스텝이라 이산화 오차가 없다. 06d 의 지수 마팅게일 $e^{\theta B_t-\theta^2t/2}$ 는 $\mu=0$, $\sigma=\theta$ 인 GBM 이다.

### 2.2 정리 1 — 브라운 다리의 공분산과 $B_T$ 로부터의 독립 (Shreve II §4.7, Lawler 2e §8.7, G&S 4e §13.6)

$X_t=B_t-\frac tTB_T$ 는 평균 0 가우스 과정이고 $\mathrm{Cov}(X_s,X_t)=\min(s,t)-\frac{st}T$ 이며, 과정 $(X_t)_{t\le T}$ 는 $B_T$ 와 독립이다. 따라서 $(B_t)_{t\le T}\mid B_T=y\overset d=X_t+\frac tTy$. 또한 $P(\max_{t\le T}X_t\ge x)=e^{-2x^2/T}$ ($x>0$); 더 일반적으로 $x\to y$ 다리 (길이 $\Delta$) 에 대해 $P(\max\ge z)=e^{-2(z-x)(z-y)/\Delta}$ ($z\ge\max(x,y)$).

가정이 왜 필요한가:
- **가우스성**: "무상관 ⇒ 독립" 을 쓰는 곳이다. $\mathrm{Cov}(X_t,B_T)=0$ 하나로 과정 전체와 $B_T$ 의 독립을 얻는 것은 $(X_{t_1},\dots,X_{t_k},B_T)$ 가 결합정규이기 때문이다. 가우스가 아닌 과정(예: 06a 의 랜덤워크) 에서는 무상관이어도 독립이 아니다.
- **$\mathrm{Cov}(B_s,B_t)=\min(s,t)$** (06b): 모든 공분산 계산의 유일한 입력. 독립·정상 증분과 $\mathrm{Var}B_t=t$ 에서 나온다.
- **최댓값 공식**은 06c 의 결합밀도 $(M_t,B_t)$ 에 조건화를 건 것이라 반사 원리의 가정(대칭·연속·강마르코프) 을 그대로 물려받는다. 드리프트가 있으면 대칭이 깨지는데, 다리의 법칙은 $\mu$ 에 무관하므로 (아래 증명 끝) 공식은 그대로 쓴다.

<details><summary>증명</summary>

**(1) 공분산.** $X$ 는 $B$ 의 선형변환이므로 가우스 과정이고 평균은 0 이다. $s\le t\le T$ 에서 $\mathrm{Cov}(B_u,B_v)=\min(u,v)$ 를 넣으면
$$\mathrm{Cov}(X_s,X_t)=\mathrm{Cov}(B_s,B_t)-\tfrac tT\mathrm{Cov}(B_s,B_T)-\tfrac sT\mathrm{Cov}(B_t,B_T)+\tfrac{st}{T^2}\mathrm{Var}B_T=s-\tfrac{ts}T-\tfrac{st}T+\tfrac{st}T=\min(s,t)-\tfrac{st}T.$$
$s=t$ 에서 $\mathrm{Var}X_t=t-t^2/T=t(T-t)/T$, 그리고 $X_0=X_T=0$ 이다.

**(2) 독립.** $\mathrm{Cov}(X_t,B_T)=\mathrm{Cov}(B_t,B_T)-\frac tT\mathrm{Var}B_T=t-\frac tTT=0$. 임의의 $t_1,\dots,t_k\le T$ 에 대해 벡터 $(X_{t_1},\dots,X_{t_k},B_T)$ 는 가우스 벡터 $(B_{t_1},\dots,B_{t_k},B_T)$ 의 선형변환이므로 결합정규이고, 결합정규에서 무상관은 독립이다. 유한차원 분포가 전부 독립이므로 과정 $(X_t)_{t\le T}$ 와 $B_T$ 가 독립이다.

**(3) 조건화.** $B_t=X_t+\frac tTB_T$ 에서 $X$ 는 $B_T$ 와 독립이므로, $B_T=y$ 로 조건화해도 $X$ 의 법칙은 변하지 않고 둘째 항만 $\frac tTy$ 로 고정된다: $(B_t)_{t\le T}\mid B_T=y\overset d=X_t+\frac tTy$.

**(4) 최댓값.** 먼저 $T=1$. 06c 의 반사 원리 결합식 $P(M_1\ge x,B_1\le b)=P(B_1\ge2x-b)=1-\Phi(2x-b)$ ($b\le x$) 를 $b$ 로 미분하면 "$M_1\ge x$ 이고 $B_1\in db$" 의 밀도가 $\phi(2x-b)$ 다. $B_1$ 의 주변밀도 $\phi(b)$ 로 나누면 $P(M_1\ge x\mid B_1=b)=\phi(2x-b)/\phi(b)=e^{-2x(x-b)}$ ($b\le x$) 이고, $b=0$ 에서 (3) 에 의해 $P(\max_{t\le1}X_t\ge x)=\phi(2x)/\phi(0)=e^{-2x^2}$. 일반 $T$: 자기유사성 $B_{Tu}\overset d=\sqrt T B_u$ 에서 $[0,T]$ 다리는 $\sqrt T\times$($[0,1]$ 다리의 시간 재조정) 이므로 $P(\max\ge x)=e^{-2(x/\sqrt T)^2}=e^{-2x^2/T}$. 일반 끝점: $x\to y$ 다리 (길이 $\Delta$) 는 $x$ 를 빼면 $0\to y-x$ 다리이고, 위의 $P(M_\Delta\ge z\mid B_\Delta=b)=e^{-2z(z-b)/\Delta}$ 에 $z\to z-x$, $b\to y-x$ 를 넣으면 $e^{-2(z-x)(z-y)/\Delta}$ — 06c·06d 의 격자 보정 공식이다.

**드리프트 무관성.** $\mu t+B_t$ 의 $B_T=y-\mu T$ 조건부 법칙은 $X_t+\frac tT(y-\mu T)+\mu t=X_t+\frac tTy$: 드리프트가 조건화에 흡수되어 다리의 법칙은 $\mu$ 에 무관하다. 그래서 06d 의 `running_max_drift` 가 같은 보정식을 썼다. $\square$

</details>

### 2.3 정리 2 — 드리프트 BM 의 최댓값·도달시간 (Ross 12e §10.5, Lawler 2e §8.8; Girsanov 는 Shreve II §5.2)

$X_t=\mu t+\sigma B_t$, $a>0$:
$$P(M_t\ge a)=P(T_a\le t)=1-\Phi\!\Big(\frac{a-\mu t}{\sigma\sqrt t}\Big)+e^{2\mu a/\sigma^2}\Big(1-\Phi\!\Big(\frac{a+\mu t}{\sigma\sqrt t}\Big)\Big).$$
$\mu>0$: $T_a\sim\mathrm{IG}(a/\mu,\ a^2/\sigma^2)$, $E T_a=a/\mu$, $\mathrm{Var}T_a=a\sigma^2/\mu^3$. $t\to\infty$: $P(T_a<\infty)=\min(1,e^{2\mu a/\sigma^2})$ (06d 정리 3 과 일치); $\mu=0$ 이면 두 항이 같아져 06c 의 $2(1-\Phi(a/\sigma\sqrt t))$.

가정이 왜 필요한가:
- **측도변환** (Cameron–Martin/Girsanov): 드리프트 BM 의 경로 확률은 BM 의 경로 확률에 $e^{\mu B_t/\sigma-\mu^2t/(2\sigma^2)}$ 를 곱한 것 — 06d 의 지수 마팅게일이 바로 이 밀도다. 드리프트가 대칭을 깨서 반사 원리를 직접 못 쓰므로 이 우회가 필요하다. 시간 $t$ 까지의 사건에는 $\mathcal F_t$ 에서의 밀도만 필요하다 (경로 전체의 측도변환은 이 과정 범위 밖).
- **$(M_t,B_t)$ 의 결합분포** (06c 반사 원리), 정리 1 의 조건부 형태 $P(M_t\ge a\mid B_t=b)=e^{-2a(a-b)/t}$: 가중치를 적분할 대상.

<details><summary>증명 (스케치)</summary>

$\sigma=1$ 로 정규화한다 ($X/\sigma$, $a/\sigma$, $\mu/\sigma$). Girsanov (Shreve II §5.2; 인용) 에 의해 $\mathcal F_t$ 위의 사건 $A$ 에 대해 $P^\mu(A)=E^0\big[e^{\mu B_t-\mu^2t/2}\,1_A\big]$ 이다. $A=\{M_t\ge a\}$ 를 $B_t=b$ 로 조건화하면, 정리 1 (4) 의 $P^0(M_t\ge a\mid B_t=b)=e^{-2a(a-b)/t}$ ($b<a$; $b\ge a$ 이면 1) 를 써서
$$P^\mu(M_t\ge a)=\int_a^\infty e^{\mu b-\mu^2t/2}\phi_t(b)\,db+\int_{-\infty}^a e^{\mu b-\mu^2t/2}e^{-2a(a-b)/t}\phi_t(b)\,db,\qquad\phi_t(b)=\tfrac1{\sqrt t}\phi(b/\sqrt t).$$
첫 적분: $e^{\mu b-\mu^2t/2}\phi_t(b)=\phi_t(b-\mu t)$ (완전제곱) 이므로 $1-\Phi\big((a-\mu t)/\sqrt t\big)$. 둘째 적분: 지수를 모으면 $-\frac1{2t}\big(b^2-2\mu tb-4ab+4a^2+\mu^2t^2\big)=-\frac{(b-2a-\mu t)^2}{2t}+2\mu a$ 이므로 $e^{2\mu a}\int_{-\infty}^a\phi_t(b-2a-\mu t)\,db=e^{2\mu a}\Phi\big((-a-\mu t)/\sqrt t\big)=e^{2\mu a}\big(1-\Phi((a+\mu t)/\sqrt t)\big)$. 둘째 항의 $e^{2\mu a}$ 가 완전제곱의 나머지에서 나오는 것을 보라 — 06d 의 $e^{2\mu b/\sigma^2}$ 와 같은 출처다.

**IG.** $\mu>0$ 에서 $F(t)=P(T_a\le t)$ 를 $t$ 로 미분하면 (계산 생략) $f(t)=\frac{a}{\sqrt{2\pi t^3}}\exp\big(-\frac{(a-\mu t)^2}{2t}\big)$ 이고, 이것은 $\mathrm{IG}(m,\lambda)$ 밀도에 $m=a/\mu$, $\lambda=a^2$ 를 넣은 것과 같다 ($\lambda(t-m)^2/(2m^2t)=(a-\mu t)^2/(2t)$). 따라서 $ET_a=m=a/\mu$ (06d 의 월드 항등식과 일치), $\mathrm{Var}T_a=m^3/\lambda=a/\mu^3$; 일반 $\sigma$ 는 $a\to a/\sigma$, $\mu\to\mu/\sigma$ 로 $\mathrm{IG}(a/\mu,a^2/\sigma^2)$, 분산 $a\sigma^2/\mu^3$.

**극한.** $t\to\infty$: $\mu>0$ 이면 첫 항 $\to1$, 둘째 항 $\to0$; $\mu<0$ 이면 첫 항 $\to0$, 둘째 항 $\to e^{2\mu a}=e^{-2|\mu|a}$. $\mu=0$: 두 항이 각각 $1-\Phi(a/\sqrt t)$. 노트북에서는 시뮬레이션으로 검증한다. $\square$

</details>

### 2.4 정리 3 — GBM 의 모멘트와 장기 거동 (Ross 12e §10.3.2, Karlin & Taylor 2e §7.4, Shreve II §3.2.7)

$S_t=S_0e^{(\mu-\sigma^2/2)t+\sigma B_t}$ 에 대해
$$E S_t=S_0e^{\mu t},\quad \mathrm{Var}S_t=S_0^2e^{2\mu t}\big(e^{\sigma^2t}-1\big),\quad \mathrm{median}(S_t)=S_0e^{(\mu-\sigma^2/2)t},\quad P(S_t>S_0)=\Phi\Big(\frac{(\mu-\sigma^2/2)\sqrt t}{\sigma}\Big).$$
장기: $\frac1t\log\frac{S_t}{S_0}\to\mu-\frac{\sigma^2}2$ a.s. 따라서 $\mu<\sigma^2/2$ 이면 $S_t\to0$ a.s. 인데도 $\mu>0$ 이면 $ES_t\to\infty$.

가정이 왜 필요한가:
- **정규 mgf** $E e^{\sigma B_t}=e^{\sigma^2t/2}$: $-\sigma^2t/2$ 보정항의 출처. 이것이 없으면 $ES_t=S_0e^{(\mu+\sigma^2/2)t}$ 가 되어 "$\mu$ = 평균 성장률" 이라는 모형의 약속이 깨진다.
- **강대수법칙** $B_t/t\to0$ a.s.: a.s. 극한의 근거. 분포 수준에서는 $B_t/t\overset d=t^{-1/2}Z\to0$ 만으로 충분하지만, "거의 모든 경로" 를 말하려면 a.s. 가 필요하다.
- **연속성·독립 증분**: 로그수익률 $\log(S_t/S_s)=(\mu-\sigma^2/2)(t-s)+\sigma(B_t-B_s)$ 가 독립·정상 — 금융 모형화의 이유 (Durrett 3e §6 의 이항 모형 극한).

<details><summary>증명</summary>

$Y=\log S_t\sim N(m,v)$, $m=\log S_0+(\mu-\sigma^2/2)t$, $v=\sigma^2t$ ($\sigma B_t\sim N(0,\sigma^2t)$).

**모멘트.** 로그정규 공식 $Ee^{kY}=e^{km+k^2v/2}$ (정규 mgf) 에서 $ES_t=e^{m+v/2}=S_0e^{(\mu-\sigma^2/2)t+\sigma^2t/2}=S_0e^{\mu t}$; $ES_t^2=e^{2m+2v}=S_0^2e^{(2\mu-\sigma^2)t+2\sigma^2t}=S_0^2e^{(2\mu+\sigma^2)t}$ 이므로 $\mathrm{Var}S_t=S_0^2e^{2\mu t}(e^{\sigma^2t}-1)$.

**중앙값·확률.** $x\mapsto e^x$ 는 단조증가이므로 $\mathrm{median}(S_t)=e^{\mathrm{median}(Y)}=e^m$. $P(S_t>S_0)=P(Y>\log S_0)=P\big(Z>-(\mu-\sigma^2/2)t/(\sigma\sqrt t)\big)=\Phi\big((\mu-\sigma^2/2)\sqrt t/\sigma\big)$.

**a.s. 극한.** $\frac1t\log\frac{S_t}{S_0}=(\mu-\frac{\sigma^2}2)+\sigma\frac{B_t}t$. 정수 $n$ 에서 $B_n=\sum_{k=1}^n(B_k-B_{k-1})$ 은 i.i.d. $N(0,1)$ 의 합이므로 SLLN 으로 $B_n/n\to0$ a.s. 정수 사이는 $D_n=\max_{n\le t\le n+1}|B_t-B_n|$ 이 i.i.d. 이고 $E D_n^2<\infty$ ($D_n\overset d=\max_{s\le1}|B_s|$ (독립 증분·정상성), 06c 의 반사 원리로 $M_1\overset d=|B_1|$ 이고 $\max_{s\le1}|B_s|\le M_1+\max_{s\le1}(-B_s)$ 에서 $E D_n^2\le4EM_1^2<\infty$) 이므로 $\sum_nP(D_n>\epsilon n)\le\sum_nE D_n^2/(\epsilon n)^2<\infty$ 에서 Borel–Cantelli 로 $D_n/n\to0$ a.s.; 따라서 $B_t/t\to0$ a.s. 그러므로 $\mu-\sigma^2/2<0$ 이면 $\log S_t\to-\infty$, 즉 $S_t\to0$ a.s. 한편 $ES_t=S_0e^{\mu t}$ 는 $\mu>0$ 이면 발산한다 — 극한과 기댓값이 교환되지 않는 예다 (06d E3 와 같은 현상). $\square$

</details>

### 2.5 직관

**다리는 '도착지를 아는 BM'** 이다. 미래를 알면 현재의 불확실성이 $t$ 에서 $t(T-t)/T$ 로 줄고, 중간이 가장 불확실하다 (양 끝은 못 박혀 있다). 06c 에서 격자 사이를 정확히 채운 도구가 바로 이것이고, 조건화 표현은 "BM = 다리 + 직선" 이라는 분해다: 직선 $\frac tTB_T$ 가 $B_T$ 의 정보 전부를 담고, 다리는 그것과 독립인 나머지다.

**드리프트는 대칭을 깨서** 반사 원리를 못 쓰게 하지만, 06d 의 지수 마팅게일 $e^{\mu B_t-\mu^2t/2}$ 가 '드리프트 없는 세계의 확률에 곱하는 환율' 이 되어 공식을 준다. 두 항 공식의 첫 항은 "그냥 $B_t$ 가 $a$ 를 넘은 경로", 둘째 항은 "넘었다가 돌아온 경로" 인데, 후자는 반사되어 $2a-b$ 근처에 있으므로 환율이 $e^{2\mu a}$ 만큼 다르다. $\mu>0$ 이면 $T_a$ 의 평균이 $a/\mu$ 로 유한해지고 (역가우스), $\mu<0$ 이면 애초에 못 닿을 확률 $1-e^{-2|\mu|a}$ 가 남는다.

**GBM 은 '로그수익률이 BM'** 인 모형이다. 평균은 오른쪽 꼬리가 끌어올리고 ($e^{\mu t}$), 전형적인 경로는 $e^{(\mu-\sigma^2/2)t}$ 를 따른다. $\sigma^2/2$ 의 **변동성 끌림**(volatility drag) 은 젠센 부등식 ($E e^Y>e^{EY}$) 의 크기이고, 07b 이토 공식의 $\frac12f''$ 항이며, 06b 의 $dB^2=dt$ 가 낳은 값이다. $\mu<\sigma^2/2$ 이면 거의 모든 경로가 0 으로 가는데 평균은 발산한다 — 극소수의 폭등 경로가 평균을 떠받치기 때문이다. 같은 이유로 그 평균을 표본평균으로 추정하려 하면 한두 경로가 결과를 지배한다.

### 2.6 함정 (traps)

- `brownian_bridge` 로 만든 다리는 격자점에서 정확히 다리 분포이고 격자 사이도 (같은 $\Delta t$ 의) 브라운 다리이므로 06c 의 다리 최댓값 보정이 그대로 적용된다. 격자 최댓값만 쓰면 $P(\max X\ge1)$ 이 $e^{-2}=0.135$ 대신 0.114 ($n=200$) 로 나온다 (E2).
- 다리 공분산 $\min(s,t)-st/T$ 에서 $T$ 를 빠뜨리면 ($T\ne1$) 틀린다. $T=2$ 에서 $\mathrm{Var}X_{1/2}=0.5\cdot1.5/2=0.375$ 이지 $0.25$ 가 아니다 (E1).
- 드리프트 최댓값 공식의 둘째 항 $e^{2\mu a/\sigma^2}$: $\mu<0$ 이면 1 보다 작아 $t\to\infty$ 극한이 $e^{-2|\mu|a/\sigma^2}<1$; $\mu>0$ 이면 첫 항이 1 로 가고 둘째 항은 0 으로 간다. 지수의 부호를 바꾸면 확률이 1 을 넘는다 — 부호 검산은 "$\mu>0$, $t\to\infty$ 에서 1 이 되는가".
- GBM 의 평균 $S_0e^{\mu t}$ 는 로그정규 오른쪽 꼬리가 만든다. $\sigma^2t$ 가 크면 표본평균의 SE 가 폭발한다 ($\sigma=0.4$, $t=50$: 참 sd $6.65\times10^4$, 상대 SE 39% at $N=20000$). 게다가 **표본 sd 도 참 sd 를 크게 과소추정**하므로 se 막대조차 믿을 수 없다. 평균 대신 로그의 평균이나 분위수·확률을 검증하라.
- $\mu-\sigma^2/2<0$ 이면 $S_t\to0$ a.s. 이지만 어떤 유한 $t$ 에서도 $P(S_t>S_0)>0$ ($t=50$, $\sigma=0.4$: 0.298). "거의 확실히 0 으로 간다" 는 유한 시간의 진술이 아니다.
- `gbm_paths` 는 정확한 로그정규 스텝이라 이산화 편향이 없다. 07c 의 Euler–Maruyama 와 혼동하지 말 것 — 거기서는 $\Delta t$ 편향이 주제다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 왜 $E[S_t]$ 는 커지는데 대부분의 경로는 0 으로 갈 수 있는지, 로그정규 꼬리를 언급하며 세 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $\frac{t(T-t)}T$, $\min(s,t)-\frac{st}T$, 두 항 공식 ($\Phi(0)=\frac12$, $1-\Phi(2)=0.02275$, $e^2=7.389$), $S_0e^{\mu t}$, $S_0e^{(\mu-\sigma^2/2)t}$, $\Phi\big((\mu-\sigma^2/2)\sqrt t/\sigma\big)$.

In [ ]:
predictions = {
    # T = 1 표준 다리: Var(X_{1/2}) 는?
    "var_bridge_half": None,
    # 같은 다리: Cov(X_{1/4}, X_{3/4}) 는?
    "cov_bridge_quarters": None,
    # 드리프트 mu = 1, sigma = 1, a = 1: P(T_1 <= 1) 은? (드리프트 없을 때 0.3173 과 비교)
    "p_T1_le_1_drift1": None,
    # GBM S0 = 100, mu = 0.05, sigma = 0.3: E[S_2] 는?
    "e_S2": None,
    # 같은 설정에서 S_2 의 중앙값은? (평균보다 큰가 작은가?)
    "median_S2": None,
    # 같은 설정에서 P(S_2 > 100) 은?
    "p_S2_gt_S0": None,
    # sigma = 0.4 로 바꾸고 t = 50: P(S_50 > 100) 은?
    "p_S50_gt_S0_sigma04": None,
    # 같은 설정에서 E[S_50] 은?
    "e_S50": None,
    # 같은 설정에서 S_50 의 중앙값은? (평균과의 비율에 주목)
    "median_S50": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서만 쓰는 도우미를 정의한다. `exit_times` 는 06d 와 같다 (드리프트 BM 의 $(-a,b)$ 탈출; `a=np.inf` 이면 아래 장벽 없음; 다리 교차 보정과 스텝 중점 시각). 오늘은 $a=\infty$, $b=1$ 로 도달시간 $T_1$ 을 얻는 데 쓴다.

In [ ]:
# helper (spkit에 없어서 여기서 정의) — 06d 와 동일
def exit_times(a, b, mu, sigma, dt, n_paths, rng, t_max=40.0, bridge=True):
    """X_t = mu t + sigma B_t 의 (-a, b) 탈출. 경로 벡터화, 활성 경로만 갱신. 반환 (hit_b, tau, x_end)."""
    x = np.zeros(n_paths)
    tau = np.full(n_paths, np.inf)
    hit_b = np.zeros(n_paths, dtype=bool)
    active = np.ones(n_paths, dtype=bool)
    sd = sigma * np.sqrt(dt)
    for k in range(int(round(t_max / dt))):
        idx = np.flatnonzero(active)
        if idx.size == 0:
            break
        xo = x[idx]
        xn = xo + mu * dt + sd * rng.standard_normal(idx.size)
        up, dn = xn >= b, xn <= -a
        if bridge:  # 격자점은 안에 있어도 스텝 도중에 넘었을 확률 (정리 1 따름정리)
            p_up = np.exp(-2 * (b - xo) * (b - xn) / (sigma**2 * dt))
            p_dn = np.exp(-2 * (a + xo) * (a + xn) / (sigma**2 * dt)) if np.isfinite(a) else 0.0
            up |= (~dn) & (rng.random(idx.size) < p_up)
            dn |= (~up) & (rng.random(idx.size) < p_dn)
        done = up | dn
        tau[idx[done]] = (k + 0.5) * dt if bridge else (k + 1) * dt
        hit_b[idx[done]] = up[done]
        active[idx[done]] = False
        x[idx[~done]] = xn[~done]
    return hit_b, tau, x

`running_max_drift` 는 06d 의 다리 보정 러닝 최댓값에 `save_at` (스냅샷을 저장할 스텝 인덱스) 를 더한 것이고, `bridge_step_max` 는 06c 의 구간별 최댓값 표본기 (E2 용), `drift_max_cdf` 는 정리 2 의 두 항 공식, `cond_var_exact` 는 4.2 의 정확 조건부 2 차 모멘트다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def running_max_drift(mu, sigma, dt, n_steps, n_paths, rng, save_at=()):
    """드리프트 BM 의 다리 보정 러닝 최댓값 M, 끝값 x, 그리고 {step: M 스냅샷}."""
    x, M, snaps = np.zeros(n_paths), np.zeros(n_paths), {}
    sd = sigma * np.sqrt(dt)
    for k in range(1, n_steps + 1):
        xn = x + mu * dt + sd * rng.standard_normal(n_paths)
        U = rng.random(n_paths)
        M = np.maximum(M, 0.5 * (x + xn + np.sqrt((xn - x) ** 2 - 2 * sigma**2 * dt * np.log(U))))
        x = xn
        if k in save_at:
            snaps[k] = M.copy()
    return M, x, snaps


def bridge_step_max(X, dt, rng):
    """격자 구간별 정확한 최댓값 표본 (양 끝점 조건부 브라운 다리; shape (n_paths, n_steps)) — 06c."""
    x, y = X[:, :-1], X[:, 1:]
    U = rng.random(x.shape)
    return 0.5 * (x + y + np.sqrt((y - x) ** 2 - 2 * dt * np.log(U)))


def drift_max_cdf(a, mu, sigma, t):
    """P(M_t >= a) = P(T_a <= t) for X = mu t + sigma B (정리 2, 두 항 공식)."""
    t = np.asarray(t, dtype=float)
    return norm.sf((a - mu * t) / (sigma * np.sqrt(t))) + np.exp(2 * mu * a / sigma**2) * norm.sf((a + mu * t) / (sigma * np.sqrt(t)))


def cond_var_exact(eps):
    """E[B_{1/2}^2 | |B_1| < eps] = 1/4 + (1/4) E[Z^2 | |Z| < eps]  (다리 분산 + 직선항의 분산)."""
    return 0.25 + 0.25 * quad(lambda z: z**2 * norm.pdf(z), -eps, eps)[0] / (2 * norm.cdf(eps) - 1)

### 4.1 브라운 다리 — 포락선과 공분산

$[0,1]$, $\Delta t=0.01$ 이므로 $t=1/4,1/2,3/4$ 는 열 25, 50, 75 다. 정리 1: $E[X_{1/2}^2]=1/4$, $E[X_{1/4}X_{3/4}]=1/4-3/16=1/16$. 성긴 격자 ($\Delta t=0.1$) 에서 경험 공분산 행렬 전체를 $\min(s,t)-st$ 와 비교해 최대 오차도 본다.

In [ ]:
t, X = brownian_bridge(N_PATHS, 100, 1.0, rng)     # dt = 0.01
var_half_hat = mc_estimate(X[:, 50] ** 2)
cov_q_hat = mc_estimate(X[:, 25] * X[:, 75])
s = t[::10]
C_hat, C_exact = np.cov(X[:, ::10].T), np.minimum.outer(s, s) - np.outer(s, s)
print(f"ends pinned: max|X_0| = {np.abs(X[:, 0]).max():.1e}, max|X_1| = {np.abs(X[:, -1]).max():.1e}")
print(f"E[X_1/2^2]     = {var_half_hat}   (exact t(1-t) = {0.5 * 0.5:.4f})")
print(f"E[X_1/4 X_3/4] = {cov_q_hat}   (exact 1/4 - 3/16 = {0.25 - 3 / 16:.4f})")
print(f"max |C_hat - C_exact| on the 11x11 grid = {np.abs(C_hat - C_exact).max():.4f}  (largest entry 0.25)")

왼쪽: 10 경로와 포락선 $\pm2\sqrt{t(1-t)}$ — 양 끝이 0 에 못 박히고 중간이 가장 넓다. 오른쪽: 경험 분산 `X.var(0)` 이 곡선 $t(1-t)$ 위에 놓인다 (BM 의 $t$ 와 비교).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
plot_paths(t, X, ax=axes[0], n_show=10, alpha=0.7)
env = 2 * np.sqrt(t * (1 - t))
axes[0].plot(t, env, "k--", lw=1.2, label="±2 sqrt(t(1-t))"); axes[0].plot(t, -env, "k--", lw=1.2)
axes[0].set_xlabel("t"); axes[0].set_title("10 Brownian bridges on [0, 1]"); axes[0].legend(loc="upper left")
axes[1].plot(t, X.var(0), "o", ms=3, label=f"empirical Var(X_t), N = {N_PATHS}")
axes[1].plot(t, t * (1 - t), "-", label="t(1-t)  (bridge)")
axes[1].plot(t, t, ":", color="grey", label="t  (free BM)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("variance"); axes[1].set_title("variance of the bridge"); axes[1].legend()
plt.tight_layout(); plt.show()

### 4.2 조건화 관점 — $|B_1|<0.1$ 인 BM 경로만 남기기

정리 1 의 조건화 표현을 '채택-기각' 으로 확인한다: 자유 BM $N_{\rm big}$ 경로 중 $|B_1|<\epsilon=0.1$ 인 것만 남기면 (채택률 $2\Phi(0.1)-1=0.0797$) 남은 경로는 다리 $+\frac tTB_1$ 이어야 한다. $t=1/2$ 에서 2 차 모멘트의 정확값은 $\frac14+\frac14E[Z^2\mid|Z|<\epsilon]=0.250832$ — 다리 분산 $1/4$ 에 직선항 $\frac12B_1$ 의 분산이 더해진 것으로, $\epsilon\to0$ 에서 $1/4$ 로 간다.

In [ ]:
EPS = 0.1
t2, B = brownian_paths(N_BIG, 50, 1.0, rng)       # dt = 0.02: t = 1/2 → 열 25
keep = np.abs(B[:, -1]) < EPS
p_acc = mc_proportion(keep)
cond_var_hat = mc_estimate(B[keep, 25] ** 2)
cond_var_ex = cond_var_exact(EPS)
print(f"accepted {keep.sum()} of {N_BIG}: rate {p_acc}   (exact 2Φ(0.1)-1 = {2 * norm.cdf(EPS) - 1:.4f})")
print(f"E[B_1/2^2 | |B_1| < 0.1] = {cond_var_hat}   (exact {cond_var_ex:.6f}; pure bridge 0.25)")

왼쪽: 채택된 경로의 $B_{1/2}$ 히스토그램과 $N(0,1/4)$ 밀도. 오른쪽: 채택 경로 10 개 (회색) 와 `brownian_bridge` 로 만든 다리 10 개 (색) — 구별할 수 없어야 한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
xg = np.linspace(-2, 2, 300)
plot_hist_vs_pdf(B[keep, 25], xg, norm.pdf(xg, 0, 0.5), ax=axes[0], bins=40, label_pdf="N(0, 1/4) pdf")
axes[0].set_xlabel("B_1/2 given |B_1| < 0.1"); axes[0].set_title("conditioned BM at t = 1/2"); axes[0].legend()
for row in B[keep][:10]:
    axes[1].plot(t2, row, color="grey", lw=1, alpha=0.6)
axes[1].plot([], [], color="grey", label="BM with |B_1| < 0.1")
plot_paths(t, X, ax=axes[1], n_show=10, alpha=0.8)
axes[1].plot([], [], color="C0", label="constructed bridges")
axes[1].set_xlabel("t"); axes[1].set_title("conditioned BM (grey) vs bridges (color)"); axes[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

### 4.3 드리프트 BM 의 도달시간 — $\mu=\sigma=1$, $a=1$

`exit_times(np.inf, 1, ...)` 로 $T_1$ 을 다리 보정과 함께 얻는다 ($\Delta t=0.002$, 중점 시각이라 오차 $\le0.001$). 지평 $t_{\max}=10$ 을 넘는 경로는 $P(T_1>10)=0.00035$ 로 드물고, 그 경로는 $\tau=\infty$ 로 남으므로 평균은 $E[T_1\wedge10]$ 과 비교한다 ($E T_1=1$ 과의 차이는 $0.0006$). 검사: $P(T_1\le1)$ (정리 2 공식 = `invgauss(1,1).cdf(1)`) 과 $E[T_1\wedge10]$ (IG 로 수치적분).

In [ ]:
A3, MU3, SIG3, T_MAX3 = 1.0, 1.0, 1.0, 10.0
_, tau, _ = exit_times(np.inf, A3, MU3, SIG3, 0.002, N_PATHS, rng, t_max=T_MAX3, bridge=True)
p_T1_hat = mc_proportion(tau <= 1.0)
e_T1_hat = mc_estimate(np.minimum(tau, T_MAX3))
ig = invgauss(mu=(A3 / MU3) / (A3**2 / SIG3**2), scale=A3**2 / SIG3**2)   # T_a ~ IG(a/mu, a^2/sigma^2)
p_T1_exact = drift_max_cdf(A3, MU3, SIG3, 1.0)
e_T1_exact = ig.expect(lambda u: min(u, T_MAX3))
print(f"P(T_1 <= 1)     = {p_T1_hat}   (formula {p_T1_exact:.6f} = IG cdf {ig.cdf(1.0):.6f}; driftless {2 * norm.sf(1.0):.4f})")
print(f"E[min(T_1, 10)] = {e_T1_hat}   (exact {e_T1_exact:.6f}; E[T_1] = a/mu = {ig.mean():.1f}, Var = {ig.var():.1f})")
print(f"paths with T_1 > 10: {np.isinf(tau).sum()}   (expected {N_PATHS * ig.sf(T_MAX3):.1f})")

왼쪽: $T_1$ 히스토그램과 $\mathrm{IG}(1,1)$ 밀도 — 06c 의 $t^{-3/2}$ 멱꼬리가 지수 꼬리로 바뀌어 평균이 유한해졌다. 오른쪽: 경험 CDF (5 개 시각) 와 두 항 공식 곡선; 점선은 드리프트 없는 $2(1-\Phi(1/\sqrt t))$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
tg = np.linspace(0.01, 5, 400)
plot_hist_vs_pdf(tau[np.isfinite(tau) & (tau <= 5)], tg, ig.pdf(tg), ax=axes[0], bins=60, label_pdf="IG(1, 1) pdf")
axes[0].set_xlabel("T_1 for X_t = t + B_t"); axes[0].set_title("hitting time with drift 1"); axes[0].legend()
t_pts = np.array([0.25, 0.5, 1.0, 2.0, 4.0])
cdf_hat = [mc_proportion(tau <= tt) for tt in t_pts]
tc = np.linspace(0.01, 5, 300)
axes[1].plot(tc, drift_max_cdf(A3, MU3, SIG3, tc), "-", label="two-term formula, mu = 1")
axes[1].plot(tc, 2 * norm.sf(A3 / np.sqrt(tc)), "k--", lw=1, label="driftless 2(1 - Phi(1/sqrt t))")
axes[1].errorbar(t_pts, [r.mean for r in cdf_hat], yerr=[1.96 * r.se for r in cdf_hat], fmt="o", capsize=3, label="MC (95% CI)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("P(T_1 <= t)"); axes[1].set_title("CDF of T_1"); axes[1].legend()
plt.tight_layout(); plt.show()

### 4.4 드리프트 BM 의 최댓값 — 음의 드리프트는 포화, 양의 드리프트는 1

$a=0.5$, $\sigma=1$, $\Delta t=0.01$, 200 스텝 ($t=2$). 한 루프에서 스텝 50, 100, 200 ($t=0.5,1,2$) 의 러닝 최댓값을 저장해 $\hat P(M_t\ge0.5)$ 를 정확 곡선 위에 찍는다. $\mu=-0.5$ 는 $e^{2\mu a}=e^{-0.5}=0.6065$ 로 포화하고 (06d 의 $\sup X\sim\mathrm{Exp}(1)$), $\mu=+0.5$ 는 1 로 간다.

In [ ]:
A4, SIG4, DT4, NSTEP4 = 0.5, 1.0, 0.01, 200
SAVE = {50: 0.5, 100: 1.0, 200: 2.0}          # step index -> t
runmax_hat = {}
for mu4, n4 in [(-0.5, N_PATHS), (0.5, N_PATHS // 2)]:
    M, _, snaps = running_max_drift(mu4, SIG4, DT4, NSTEP4, n4, rng, save_at=tuple(SAVE))
    runmax_hat[mu4] = {SAVE[k]: mc_proportion(v >= A4) for k, v in snaps.items()}
    for tt, r in runmax_hat[mu4].items():
        print(f"mu = {mu4:+.1f}, t = {tt}: P(M_t >= 0.5) = {r}   (exact {drift_max_cdf(A4, mu4, SIG4, tt):.4f})")
p_M2_hat = runmax_hat[-0.5][2.0]
p_M2_exact = drift_max_cdf(A4, -0.5, SIG4, 2.0)
print(f"limits t -> inf: mu = -0.5: e^(2 mu a) = {np.exp(2 * -0.5 * A4):.4f};  mu = +0.5: 1")

In [ ]:
tc = np.linspace(0.01, 8, 400)
fig, ax = plt.subplots(figsize=(7, 4))
for mu4, col in [(-0.5, "C0"), (0.5, "C1")]:
    ax.plot(tc, drift_max_cdf(A4, mu4, SIG4, tc), "-", color=col, label=f"exact, mu = {mu4:+.1f}")
    pts = runmax_hat[mu4]
    ax.errorbar(list(pts), [r.mean for r in pts.values()], yerr=[1.96 * r.se for r in pts.values()],
                fmt="o", color=col, capsize=3, label=f"MC, mu = {mu4:+.1f}")
ax.axhline(np.exp(2 * -0.5 * A4), color="C0", ls=":", lw=1, label="limit exp(-0.5) = 0.607 (mu = -0.5)")
ax.axhline(1.0, color="C1", ls=":", lw=1)
ax.set_xlabel("t"); ax.set_ylabel("P(M_t >= 0.5)"); ax.set_title("running maximum of mu t + B_t"); ax.legend(loc="lower right")
plt.show()

### 4.5 GBM 의 주변분포 — $S_0=100$, $\mu=0.05$, $\sigma=0.3$, $T=2$

정리 3: $ES_2=100e^{0.1}=110.517$, 중앙값 $100e^{0.01}=101.005$, $P(S_2>100)=\Phi(0.02357)=0.5094$, $\log S_2\sim N(4.61517,\ 0.18)$. 평균은 검증 가능하지만 ($\mathrm{sd}=49.1$, se $0.35$) 로그의 두 모멘트가 훨씬 정밀한 검사임을 눈여겨보라.

In [ ]:
S0, MU5, SIG5, T5 = 100.0, 0.05, 0.3, 2.0
t5, S = gbm_paths(S0, MU5, SIG5, N_GBM, 100, T5, rng)
ST = S[:, -1]
L = np.log(ST)
m_log, v_log = np.log(S0) + (MU5 - SIG5**2 / 2) * T5, SIG5**2 * T5
e_S2_hat, p_S2_hat = mc_estimate(ST), mc_proportion(ST > S0)
e_logS2_hat, var_logS2_hat = mc_estimate(L), mc_estimate((L - m_log) ** 2)
print(f"E[S_2]        = {e_S2_hat}   (exact S0 e^(mu T) = {S0 * np.exp(MU5 * T5):.4f})")
print(f"median(S_2)   = {np.median(ST):.4f}   (exact S0 e^((mu - s^2/2) T) = {np.exp(m_log):.4f})")
print(f"P(S_2 > 100)  = {p_S2_hat}   (exact {norm.cdf((MU5 - SIG5**2 / 2) * np.sqrt(T5) / SIG5):.6f})")
print(f"E[log S_2]    = {e_logS2_hat}   (exact {m_log:.6f})")
print(f"Var(log S_2)  = {var_logS2_hat}   (exact sigma^2 T = {v_log:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
plot_paths(t5, S, ax=axes[0], n_show=10, alpha=0.6)
axes[0].plot(t5, S0 * np.exp(MU5 * t5), "k-", lw=2, label="mean S0 exp(mu t)")
axes[0].plot(t5, S0 * np.exp((MU5 - SIG5**2 / 2) * t5), "k--", lw=2, label="median S0 exp((mu - s^2/2) t)")
axes[0].set_xlabel("t"); axes[0].set_title("10 GBM paths, mu = 0.05, sigma = 0.3"); axes[0].legend(loc="upper left")
sg = np.linspace(20, 300, 400)
ln = lognorm(s=SIG5 * np.sqrt(T5), scale=S0 * np.exp((MU5 - SIG5**2 / 2) * T5))
plot_hist_vs_pdf(ST, sg, ln.pdf(sg), ax=axes[1], bins=60, label_pdf="log-normal pdf")
axes[1].axvline(S0 * np.exp(MU5 * T5), color="k", label="mean"); axes[1].axvline(np.exp(m_log), color="k", ls="--", label="median")
axes[1].set_xlabel("S_2"); axes[1].set_title("S_2: right-skewed"); axes[1].legend()
lg = np.linspace(3, 6.2, 400)
plot_hist_vs_pdf(L, lg, norm.pdf(lg, m_log, np.sqrt(v_log)), ax=axes[2], bins=60, label_pdf=f"N({m_log:.3f}, {v_log:.2f}) pdf")
axes[2].set_xlabel("log S_2"); axes[2].set_title("log S_2: Gaussian"); axes[2].legend()
plt.tight_layout(); plt.show()

### 4.6 변동성 끌림 — $\sigma=0.4$, $T=50$: 평균은 오르고 경로는 내린다

$\mu-\sigma^2/2=0.05-0.08=-0.03<0$. 정리 3: $ES_{50}=100e^{2.5}=1218$ (상승) 인데 중앙값은 $100e^{-1.5}=22.3$ (하강), $P(S_{50}>100)=\Phi(-0.5303)=0.298$, $E\log S_{50}=\log100-1.5=3.1052$. 평균의 참 sd 는 $6.65\times10^4$ 라 $N=20000$ 에서도 se $\approx470$ (39%) — 표에만 올리고 assert 하지 않는다. 표본평균의 얼마가 상위 1% 경로에서 오는지도 센다.

In [ ]:
SIG6, T6 = 0.4, 50.0
t6, S50 = gbm_paths(S0, MU5, SIG6, N_PATHS, 500, T6, rng)
S50T = S50[:, -1]
p_S50_hat, e_logS50_hat, e_S50_hat = mc_proportion(S50T > S0), mc_estimate(np.log(S50T)), mc_estimate(S50T)
m_log50 = np.log(S0) + (MU5 - SIG6**2 / 2) * T6
sd_true = S0 * np.exp(MU5 * T6) * np.sqrt(np.exp(SIG6**2 * T6) - 1)
top1 = np.sort(S50T)[-max(1, N_PATHS // 100):].sum() / S50T.sum()
print(f"P(S_50 > 100) = {p_S50_hat}   (exact {norm.cdf((MU5 - SIG6**2 / 2) * np.sqrt(T6) / SIG6):.6f})")
print(f"E[log S_50]   = {e_logS50_hat}   (exact {m_log50:.6f});  sample median {np.median(S50T):.2f} (exact {np.exp(m_log50):.2f})")
print(f"E[S_50]       = {e_S50_hat}   (exact {S0 * np.exp(MU5 * T6):.2f}; TRUE sd {sd_true:.3g} -> true se {sd_true / np.sqrt(N_PATHS):.0f}, "
      f"sample sd only {S50T.std(ddof=1):.3g})")
print(f"share of the sample mean carried by the top 1% of paths: {top1:.1%}")

왼쪽: 30 경로 (log-y) 와 평균 곡선 (상승), 중앙값 곡선 (하강) — 경로 다발은 중앙값을 따라 내려가고, 평균은 다발 위로 떠난다. 오른쪽: $S_0$ 위에 있는 경로 비율 vs $t$ 와 정확 곡선 $\Phi\big((\mu-\sigma^2/2)\sqrt t/\sigma\big)$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
plot_paths(t6, S50, ax=axes[0], n_show=30, alpha=0.35, lw=0.7)
axes[0].plot(t6, S0 * np.exp(MU5 * t6), "k-", lw=2, label="mean S0 exp(0.05 t)")
axes[0].plot(t6, S0 * np.exp((MU5 - SIG6**2 / 2) * t6), "k--", lw=2, label="median S0 exp(-0.03 t)")
axes[0].set_yscale("log"); axes[0].set_xlabel("t"); axes[0].set_title("30 GBM paths, mu = 0.05, sigma = 0.4 (log scale)")
axes[0].legend(loc="lower left")
axes[1].plot(t6[1:], (S50[:, 1:] > S0).mean(0), "-", label="fraction of paths above S0 (MC)")
axes[1].plot(t6[1:], norm.cdf((MU5 - SIG6**2 / 2) * np.sqrt(t6[1:]) / SIG6), "k--", label="Phi((mu - s^2/2) sqrt(t) / s)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("P(S_t > S_0)"); axes[1].set_title("volatility drag: mu - sigma^2/2 = -0.03"); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 전부 정리 1–3 의 식으로 코드에서 계산한다. 마지막 행 ($E[S_{50}]$, $\sigma=0.4$) 은 로그정규 꼬리 시연이라 표에만 보이고 assert 하지 않는다 — 'investigate' 가 나와도 (그리고 se 막대가 참 se 470 보다 훨씬 작아도) 놀라지 말 것.

In [ ]:
P = predictions
p_S2_exact = norm.cdf((MU5 - SIG5**2 / 2) * np.sqrt(T5) / SIG5)
p_S50_exact = norm.cdf((MU5 - SIG6**2 / 2) * np.sqrt(T6) / SIG6)
rows = [
    {"name": "E[X_1/2^2] (bridge, T = 1)", "predicted": P["var_bridge_half"], "estimate": var_half_hat, "exact": 0.5 * (1 - 0.5)},
    {"name": "E[X_1/4 X_3/4] (bridge)", "predicted": P["cov_bridge_quarters"], "estimate": cov_q_hat, "exact": min(0.25, 0.75) - 0.25 * 0.75},
    {"name": "E[B_1/2^2 | |B_1| < 0.1] (conditioned BM)", "predicted": None, "estimate": cond_var_hat, "exact": cond_var_ex},
    {"name": "P(T_1 <= 1), mu = sigma = 1", "predicted": P["p_T1_le_1_drift1"], "estimate": p_T1_hat, "exact": p_T1_exact},
    {"name": "E[min(T_1, 10)], mu = sigma = 1", "predicted": None, "estimate": e_T1_hat, "exact": e_T1_exact},
    {"name": "P(M_2 >= 0.5), mu = -0.5", "predicted": None, "estimate": p_M2_hat, "exact": p_M2_exact},
    {"name": "E[S_2]", "predicted": P["e_S2"], "estimate": e_S2_hat, "exact": S0 * np.exp(MU5 * T5)},
    {"name": "P(S_2 > 100)", "predicted": P["p_S2_gt_S0"], "estimate": p_S2_hat, "exact": p_S2_exact},
    {"name": "E[log S_2]", "predicted": None, "estimate": e_logS2_hat, "exact": m_log},
    {"name": "E[(log S_2 - m)^2] = Var(log S_2)", "predicted": None, "estimate": var_logS2_hat, "exact": v_log},
    {"name": "P(S_50 > 100), sigma = 0.4", "predicted": P["p_S50_gt_S0_sigma04"], "estimate": p_S50_hat, "exact": p_S50_exact},
    {"name": "E[log S_50], sigma = 0.4", "predicted": None, "estimate": e_logS50_hat, "exact": m_log50},
    {"name": "E[S_50], sigma = 0.4  [table only: heavy tail]", "predicted": P["e_S50"], "estimate": e_S50_hat, "exact": S0 * np.exp(MU5 * T6)},
]
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **E[X_1/2^2], E[X_1/4 X_3/4]**: `brownian_bridge` 의 $-\frac tTB_T$ 항이 빠졌거나 (자유 BM 이면 $0.5$, $0.25$ 가 나온다), 공분산 식에서 $st/T$ 를 잘못 썼다.
- **E[B_1/2^2 | |B_1|<0.1]**: 조건화 표현 "다리 + 직선" 이 틀렸거나 (순수 다리값 $0.25$ 와는 $0.2$ SE 차이라 구별 못 함), 채택 조건의 열 인덱스가 $t=1$ 이 아니다.
- **P(T_1 <= 1), E[min(T_1, 10)]**: 두 항 공식의 부호 ($e^{+2\mu a}$) 나 `exit_times` 의 다리 보정이 틀렸다; 격자만 쓰면 $P(T_1\le1)$ 이 약 $0.01$ 작게 나온다 (밀린 장벽 $1+0.5826\,\sigma\sqrt{\Delta t}=1.026$ → $0.656$).
- **P(M_2 >= 0.5), mu = -0.5**: `running_max_drift` 의 보정식 ($\sigma^2\Delta t$) 또는 스냅샷 인덱스 (스텝 200 = $t=2$) 오류.
- **E[S_2], P(S_2 > 100), E[log S_2], Var(log S_2)**: `gbm_paths` 의 $-\sigma^2t/2$ 보정이 빠지면 $E[S_2]$ 가 $S_0e^{(\mu+\sigma^2/2)t}=121.0$ 으로, $E\log S_2$ 는 $4.705$ 로 나온다 — 정리 3 의 첫 가정 그대로.
- **P(S_50 > 100), E[log S_50]**: 로그의 모멘트는 정밀하므로 여기서 틀리면 $\mu-\sigma^2/2$ 의 부호·크기 문제다.
- **E[S_50] [table only]**: 몇 SE 벗어나도 (또는 우연히 'ok' 가 떠도) 정보가 없다 — 표본 sd 자체가 참 sd $6.65\times10^4$ 를 과소추정하기 때문이다 (E3 에서 더 극단적으로).

In [ ]:
assert_close(var_half_hat, 0.5 * (1 - 0.5), k=4, name="Var(X_1/2) bridge")
assert_close(cov_q_hat, min(0.25, 0.75) - 0.25 * 0.75, k=4, name="Cov(X_1/4, X_3/4) bridge")
assert_close(cond_var_hat, cond_var_ex, k=4, name="E[B_1/2^2 | |B_1|<0.1]")
assert_close(p_T1_hat, p_T1_exact, k=4, name="P(T_1<=1) drift 1")
assert_close(e_T1_hat, e_T1_exact, k=4, name="E[min(T_1,10)] drift 1")
assert_close(p_M2_hat, p_M2_exact, k=4, name="P(M_2>=0.5) drift -0.5")
assert_close(e_S2_hat, S0 * np.exp(MU5 * T5), k=4, name="E[S_2]")
assert_close(p_S2_hat, p_S2_exact, k=4, name="P(S_2>100)")
assert_close(e_logS2_hat, m_log, k=4, name="E[log S_2]")
assert_close(var_logS2_hat, v_log, k=4, name="Var(log S_2)")
assert_close(p_S50_hat, p_S50_exact, k=4, name="P(S_50>100) sigma=0.4")
assert_close(e_logS50_hat, m_log50, k=4, name="E[log S_50] sigma=0.4")
# E[S_50] (sigma = 0.4) 는 assert 하지 않는다: 로그정규 꼬리 때문에 표본평균도 표본 se 도 믿을 수 없다 (z 만 출력)
print(f"E[S_50]: z = {(e_S50_hat.mean - S0 * np.exp(MU5 * T6)) / e_S50_hat.se:+.1f} sample-SE  "
      f"(= {(e_S50_hat.mean - S0 * np.exp(MU5 * T6)) / (sd_true / np.sqrt(N_PATHS)):+.2f} true-SE)")
print("all asserts passed")

## 6. 연습문제

### E1 (계산) — 정리 1–3 을 손으로

(a) $[0,2]$ 다리의 $\mathrm{Var}(X_{1/2})$ 와 $\mathrm{Var}(X_1)$. (b) GBM $S_0=1$, $\mu=0$, $\sigma=1$: $ES_1$, $\mathrm{Var}S_1$, 중앙값. (c) 드리프트 $\mu=1$, $\sigma=1$, $a=1$: $ET_1$, $\mathrm{Var}T_1$, $P(T_1<\infty)$; $\mu=-1$ 이면 $P(T_1<\infty)$ 는?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\mathrm{Var}X_t=t(T-t)/T$, $T=2$: $\mathrm{Var}X_{1/2}=0.5\cdot1.5/2=0.375$, $\mathrm{Var}X_1=1\cdot1/2=0.5$ (함정 2: $T=1$ 공식 $t(1-t)$ 를 쓰면 $0.25$, $0$ 으로 틀린다).

(b) $S_1=e^{-1/2+B_1}$ 은 06d 의 지수 마팅게일 ($\theta=1$) 이므로 $ES_1=1$; $\mathrm{Var}S_1=S_0^2e^{2\mu}(e^{\sigma^2}-1)=e-1=1.7183$; 중앙값 $e^{\mu-\sigma^2/2}=e^{-1/2}=0.6065$.

(c) $T_1\sim\mathrm{IG}(a/\mu,a^2/\sigma^2)=\mathrm{IG}(1,1)$: $ET_1=1$, $\mathrm{Var}T_1=a\sigma^2/\mu^3=1$, $P(T_1<\infty)=1$. $\mu=-1$: $P(T_1<\infty)=e^{2\mu a/\sigma^2}=e^{-2}=0.1353$ (06d 정리 3 의 한쪽 장벽과 같다).

```python
print(0.5 * 1.5 / 2, 1 * 1 / 2)                       # 0.375 0.5
print(1.0, np.e - 1, np.exp(-0.5))                    # 1.0 1.718 0.6065
print(1 / 1, 1 * 1 / 1**3, 1.0, np.exp(-2))           # 1.0 1.0 1.0 0.1353
```

</details>

### E2 (유도 후 시뮬레이션 검증) — 다리의 최댓값 $e^{-2x^2}$

06c 의 결합분포에서 $P(\max_{t\le1}X_t\ge x)=e^{-2x^2}$ (표준 다리) 를 유도하라 (정리 1 증명의 (4) 를 책을 덮고 스스로 채우기). 그런 다음 `t, X = brownian_bridge(N_PATHS, 200, 1.0, rng)` 에 `bridge_step_max` 를 적용해 `mc_proportion(Ms.max(1) >= 1)` 이 $e^{-2}=0.135335$ 와 맞는지 `assert_close` 하고, 격자 최댓값 `X.max(1) >= 1` 은 왜 0.114 근처로 낮게 나오는지 06c 의 밀린 장벽 상수 $\beta=0.5826$ 으로 설명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** 반사 원리 (06c): $b\le x$ 에서 $P(M_1\ge x,B_1\le b)=P(B_1\ge2x-b)=1-\Phi(2x-b)$. $b$ 로 미분하면 $\{M_1\ge x,\ B_1\in db\}$ 의 밀도 $\phi(2x-b)$; $B_1$ 의 밀도 $\phi(b)$ 로 나누면 $P(M_1\ge x\mid B_1=b)=\phi(2x-b)/\phi(b)=\exp\big(-\frac{(2x-b)^2-b^2}2\big)=e^{-2x(x-b)}$. 정리 1 의 조건화 표현에 의해 표준 다리는 $B\mid B_1=0$ 이므로 $b=0$: $P(\max X\ge x)=e^{-2x^2}$. ($x=1$: $e^{-2}=0.135335$.)

**격자 편향.** `brownian_bridge` 의 격자값은 정확한 다리 분포이고, 격자 사이도 다리이므로 06c 의 `bridge_step_max` 가 그대로 적용된다 — 함정 1. 격자 최댓값만 쓰면 06c 의 밀린 장벽 근사에 따라 장벽이 $1+\beta\sqrt{\Delta t}=1+0.5826\sqrt{0.005}=1.0412$ 로 밀린 셈이라 $e^{-2(1.0412)^2}=0.1144$ 가 나온다 — 관측 $0.114$ 와 일치. $\Delta t$ 를 100 배 줄여야 편향이 10 배 준다.

```python
tE, XE = brownian_bridge(N_PATHS, 200, 1.0, rng)
Ms = bridge_step_max(XE, 1 / 200, rng)
p_bridge, p_grid = mc_proportion(Ms.max(1) >= 1.0), mc_proportion(XE.max(1) >= 1.0)
beta = 0.5826
print(p_bridge, np.exp(-2))                                    # 약 0.1354 ± 0.0024 (N=20000) vs 0.135335
print(p_grid, np.exp(-2 * (1 + beta * np.sqrt(1 / 200)) ** 2))  # 약 0.1137 ± 0.0022 vs shifted-barrier 0.1144
assert_close(p_bridge, np.exp(-2), k=4, name="E2 bridge-corrected max")
```

</details>

### E3 (가정을 깨보기) — '표본평균 = 기댓값' 이 무력해질 때

GBM $S_0=100$, $\mu=0$, $\sigma=1$, $T=20$ 은 마팅게일이라 $ES_{20}=100$ 이 정확하다. `gbm_paths(100, 0.0, 1.0, N_PATHS, 200, 20.0, rng)` 로 (i) 표본평균과 그 (표본) se, (ii) 러닝 표본평균 그림, (iii) `mc_proportion(S[:,-1] > 100)`, 표본 중앙값, `mc_estimate(np.log(S[:,-1]))` 을 구하라. 어느 것이 믿을 만한가? 표본 se 는 참 se $\mathrm{sd}(S_{20})/\sqrt N$ 과 얼마나 다른가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

참값: $\mathrm{sd}(S_{20})=100\sqrt{e^{20}-1}=2.20\times10^6$ 이므로 참 se $=2.20\times10^6/\sqrt{20000}=1.56\times10^4\gg100$. 표본평균은 무의미하다. 더 나쁜 것은 **표본 sd 가 참 sd 를 수백 배 과소추정** 한다는 점이다 — 분포를 지배하는 꼬리 ($S_{20}>10^6$ 인 확률은 $P(Z>(\log10^4+10)/\sqrt{20})=P(Z>4.3)\approx10^{-5}$) 가 표본에 거의 없기 때문이다. 그래서 한 실행 ($N=20000$) 에서는 `mc_estimate` 가 대략 $25\pm5$ 를 주고 `assert_close(…, 100)` 는 $z\approx-15$ 로 실패했지만, 난수 상태에 따라 표본평균은 $17$ 에서 $190$ 까지, $z$ 는 $-20$ 에서 $+1$ 사이를 널뛴다 (30 개 시드 중 절반 정도는 통과). **통과하든 실패하든 정보가 없다** — 추정도 틀리고 오차 막대도 틀리기 때문이다. 러닝 평균 그림은 한두 경로가 도착할 때마다 계단처럼 뛴다.

반면 $P(S_{20}>100)=\Phi\big(-\sqrt{20}/2\big)=0.012674$ (se $0.0008$), 중앙값 $100e^{-10}=0.00454$, $E\log S_{20}=\log100-10=-5.39483$ (sd $\sqrt{20}=4.47$, se $0.032$) 는 모두 정밀하다. 교훈: 무거운 꼬리에서는 로그·분위수·확률로 검증하라 (07e 옵션 가격에서 다시 만난다).

```python
tE3, S3 = gbm_paths(100, 0.0, 1.0, N_PATHS, 200, 20.0, rng)
S3T = S3[:, -1]
sd_true3 = 100 * np.sqrt(np.exp(20) - 1)
print(mc_estimate(S3T), f"true se {sd_true3 / np.sqrt(N_PATHS):.3g}, sample sd {S3T.std(ddof=1):.3g} vs true {sd_true3:.3g}")
print(mc_proportion(S3T > 100), norm.cdf(-np.sqrt(20) / 2))      # 0.0127 ± 0.0008 vs 0.012674
print(np.median(S3T), 100 * np.exp(-10))                           # ≈ 0.0045 vs 0.004540
print(mc_estimate(np.log(S3T)), np.log(100) - 10)                  # -5.39 ± 0.03 vs -5.394830
assert_close(mc_proportion(S3T > 100), norm.cdf(-np.sqrt(20) / 2), k=4, name="E3 P(S_20 > 100)")
assert_close(mc_estimate(np.log(S3T)), np.log(100) - 10, k=4, name="E3 E[log S_20]")
run_mean = np.cumsum(S3T) / np.arange(1, N_PATHS + 1)
plt.plot(run_mean); plt.axhline(100, color="k", ls="--"); plt.xlabel("number of paths"); plt.ylabel("running mean of S_20"); plt.show()
```

</details>

## 7. 정리

1. 브라운 다리 $X_t=B_t-\frac tTB_T$: 평균 0, $\mathrm{Cov}=\min(s,t)-st/T$, $B_T$ 와 독립 → $B\mid B_T=y$ 는 $X_t+\frac tTy$. $P(\max X\ge x)=e^{-2x^2/T}$, 일반 끝점 $e^{-2(z-x)(z-y)/\Delta}$ — 06c·06d 격자 보정의 근거.
2. 드리프트 BM: $P(M_t\ge a)=1-\Phi\big(\frac{a-\mu t}{\sigma\sqrt t}\big)+e^{2\mu a/\sigma^2}\big(1-\Phi\big(\frac{a+\mu t}{\sigma\sqrt t}\big)\big)$ (지수 마팅게일 = 환율); $\mu>0$ 이면 $T_a\sim\mathrm{IG}(a/\mu,a^2/\sigma^2)$, $ET_a=a/\mu$.
3. GBM $S_t=S_0e^{(\mu-\sigma^2/2)t+\sigma B_t}$: 로그정규, $ES_t=S_0e^{\mu t}$, 중앙값 $S_0e^{(\mu-\sigma^2/2)t}$, $\mathrm{Var}=S_0^2e^{2\mu t}(e^{\sigma^2t}-1)$, $P(S_t>S_0)=\Phi\big((\mu-\sigma^2/2)\sqrt t/\sigma\big)$.
4. 변동성 끌림 $\sigma^2/2$: $\mu<\sigma^2/2$ 이면 $S_t\to0$ a.s. 이면서 $ES_t\to\infty$ 가 가능 — 평균은 극소수 경로가 떠받친다.
5. 무거운 꼬리(로그정규)에서는 표본평균 MC 가 무력하고 표본 se 조차 과소추정된다: 확률·분위수·로그 모멘트로 검증한다.

**Trap 카드.** Q: GBM 에서 $\mu>0$ 이면 주가는 결국 오른다? → A: 평균 $ES_t=S_0e^{\mu t}$ 는 오르지만 경로의 장기 성장률은 $\mu-\sigma^2/2$ 다. $\sigma^2/2>\mu$ 이면 거의 모든 경로가 0 으로 간다 — 평균은 극소수의 폭등 경로가 떠받친다 ($\sigma=0.4$, $\mu=0.05$, $t=50$: 평균 1218 vs 중앙값 22).

**다음 노트북: 07a** — 06b 의 $dB^2=dt$ 와 오늘의 $\sigma^2/2$ 끌림은 모두 "보통 미적분이 안 된다" 는 신호였다. 07a 에서 $\int B\,dB$ 를 리만합으로 직접 만들어 이토 적분을 정의하고 (좌단점 → $(B_T^2-T)/2$, 중점 → $B_T^2/2$) 등거리성을 검증한다.

log/progress.md 한 줄 템플릿: `- [ ] 06e 브라운 다리, 드리프트, 기하 브라운 운동 — YYYY-MM-DD, 소요 __분, Predict 적중 __/9, 막힌 곳: __`